# 11. End-to-End ML Project - Titanic (Capstone)

## What is this notebook about?

This is the **capstone notebook**. We tie everything together with a complete ML pipeline:

1. **Load** raw data
2. **Explore** with visualizations and stats
3. **Clean** missing values
4. **Engineer** features
5. **Split** into train/test
6. **Compare** 5 different models with cross-validation
7. **Tune** the best one with GridSearchCV
8. **Evaluate** on the held-out test set with multiple metrics
9. **Inspect** feature importance and individual predictions

This workflow generalizes to **any tabular ML problem**. Drop in your own dataset and 80% of the code stays the same.

## Dataset

The Titanic dataset (built into seaborn). Goal: predict whether a passenger survived.

In [ ]:
# If you're running locally, uncomment and run this once.
# In Google Colab, all of these are pre-installed - you can skip this cell.
# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebook
import numpy as np                 # numerical arrays
import pandas as pd                # tabular data (DataFrames)
import matplotlib.pyplot as plt    # plotting
import seaborn as sns              # prettier statistical plots

# Set up plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Set a random seed so results are reproducible
np.random.seed(42)

## Step 1. Load and explore

In [ ]:
# Load the Titanic dataset
df = sns.load_dataset("titanic")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Always check missing values and class balance before modelling
print("Missing values (top 5 columns):")
print(df.isnull().sum().sort_values(ascending=False).head())
print()
print("Class balance (0 = died, 1 = survived):")
print(df["survived"].value_counts(normalize=True).round(3))

**Slight imbalance** (62% died, 38% survived). Stratified split is a good idea.

## Step 2. Clean and engineer features

In [ ]:
# Pick the columns we want
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()

# Engineer new features
df["family_size"] = df["sibsp"] + df["parch"] + 1     # total family on board
df["is_alone"]    = (df["family_size"] == 1).astype(int)

# Fill missing values with sensible defaults
df["age"].fillna(df["age"].median(), inplace=True)
df["embarked"].fillna(df["embarked"].mode()[0], inplace=True)

# Convert categoricals to numbers (one-hot encoding)
df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=True)

print(f"Shape after cleaning: {df.shape}")
df.head()

## Step 3. Split into train and test

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Separate features (X) from target (y)
X = df.drop(columns=["survived"])
y = df["survived"]

# Use stratify to keep the survival rate the same in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
print(f"Training: {X_train.shape}")
print(f"Test:     {X_test.shape}")

## Step 4. Compare 5 models with 5-fold cross-validation

We'll try Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and KNN.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Each model wrapped in a Pipeline (scaler + model) - prevents leakage
models = {
    "Logistic Regression":  LogisticRegression(max_iter=1000),
    "Decision Tree":        DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":        RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting":    GradientBoostingClassifier(random_state=42),
    "KNN":                  KNeighborsClassifier(n_neighbors=7),
}

# Cross-validate each
print(f"{'Model':25s} {'CV Accuracy (mean +/- std)':30s}")
print("-" * 60)
for name, m in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", m)])
    s = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    print(f"{name:25s} {s.mean():.3f} +/- {s.std():.3f}")

**Gradient Boosting and Random Forest typically lead.** Let's tune Gradient Boosting in the next step.

## Step 5. Tune Gradient Boosting with GridSearchCV

In [ ]:
# Pipeline with the best-looking model from above
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  GradientBoostingClassifier(random_state=42)),
])

# Grid of hyperparameters to try
# Total: 3 x 3 x 3 = 27 combinations, each tested with 5-fold CV = 135 fits
param_grid = {
    "model__n_estimators":  [100, 200, 300],
    "model__learning_rate": [0.05, 0.1, 0.2],
    "model__max_depth":     [2, 3, 4],
}

# Run the search (use n_jobs=-1 to parallelize across CPU cores)
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best hyperparameters:", grid.best_params_)
print(f"Best CV accuracy:     {grid.best_score_:.3f}")

## Step 6. Final test set evaluation

This is the moment of truth. We have not touched the test set yet.

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, roc_auc_score,
)

# Use the best tuned model
best = grid.best_estimator_

# Predict on the held-out test set
y_pred  = best.predict(X_test)
y_proba = best.predict_proba(X_test)[:, 1]

# Multiple metrics give a complete picture
print(f"Test accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_proba):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

In [ ]:
# Confusion matrix shows what kind of mistakes the model makes
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greys",
            xticklabels=["Died", "Survived"],
            yticklabels=["Died", "Survived"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Final Test Set Confusion Matrix")
plt.show()

## Step 7. Feature Importance

Understand WHY the model makes its predictions.

In [ ]:
# Get the trained Gradient Boosting model from inside the pipeline
model = best.named_steps["model"]

# Plot importance (sums to 1.0)
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(8, 5))
imp.plot.barh(color="gray")
plt.title("Feature Importance - Gradient Boosting")
plt.xlabel("Importance")
plt.show()

**You'll typically see:** `sex_male`, `fare`, `age`, and `pclass` at the top. Historically: women, the wealthy, and children had higher survival rates.

## Step 8. Inspect individual predictions

Always sanity-check by looking at actual predictions vs reality.

In [ ]:
# Take 10 test samples
sample = X_test.head(10).copy()
sample["actual"]    = y_test.head(10).values
sample["predicted"] = best.predict(sample.drop(columns=["actual"]))
sample["prob_survived"] = best.predict_proba(
    sample.drop(columns=["actual", "predicted"])
)[:, 1].round(3)

# Show actual vs predicted
sample[["actual", "predicted", "prob_survived"]]

## Step 9. Save the model for later use

In [ ]:
# Save the entire pipeline (scaler + model) to disk
import joblib

joblib.dump(best, "titanic_model.pkl")
print("Saved to titanic_model.pkl")

# Loading it later:
# loaded = joblib.load("titanic_model.pkl")
# loaded.predict(new_data)

## Step 10. Where to go next

You just built a complete ML system from scratch! To level up:

1. **Try XGBoost or LightGBM** instead of GradientBoostingClassifier - usually faster and a bit more accurate.

2. **Engineer the `Title` feature** (Mr/Mrs/Miss/Master/Dr) from the original Kaggle Titanic CSV. It's often a top predictor.

3. **Try a different problem:**
   - **House Prices** (Kaggle) - regression
   - **IMDB sentiment** - text classification
   - **MNIST digits** - image classification

4. **Deploy** the model: wrap it in a Flask or FastAPI endpoint.

5. **Build a Streamlit dashboard** that takes user inputs and returns predictions.

Congratulations on completing the notebook series! You now know:
- Linear regression, logistic regression, regularization
- Decision trees, random forests, gradient boosting
- Distance metrics, K-Means, hierarchical clustering, KNN
- The complete ML workflow from data to deployment

Go build something cool. 🚢